In [2]:
import pandas as pd
import re

df = pd.read_json("/Users/chelseajaculina/GitHub/Multi-Agent-Collaboration-System/data/02-raw/k8s_combined_incidents.jsonl", lines=True)
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Shape: 7000 rows x 19 columns


,scenario_id,namespace,pod_name,service_account_name,node,pod_status,image,container_state,last_state,ready,restart_count,node_selectors,claim_name,event_reason,event_message,pod_describe,pod_logs,pod_logs_previous,evidence_text
0,pvc_not_found_mountfail,data-pipeline,busybox-pn3zirfh-86bbb7957c-7754l,default,<none>,Pending,busybox:1.36,None,None,None,NaN,<none>,missing-pvc-pn3zirfh,FailedScheduling,0/1 nodes are available: persistentvolumeclaim...,Name: busybox-pn3zirfh-86bbb7957c-...,,,POD DESCRIBE:\nName: busybox-pn3zi...
1,rbac_forbidden,monitoring,kube-state-metrics-hnw5520p-7fbb7866df-8rsgh,app-sa-hnw5520p,kind-control-plane/172.18.0.5,Running,registry.k8s.io/kube-state-metrics/kube-state-...,Running,None,True,0.0,<none>,None,Scheduled,Successfully assigned monitoring/kube-state-me...,Name: kube-state-metrics-hnw5520p-...,I0331 06:37:30.545502 1 wrapper.go:120] ...,Error from server (BadRequest): previous termi...,POD DESCRIBE:\nName: kube-state-me...
2,rbac_forbidden,qa-app,ingress-nginx-4g3nkfre-656c6b79b6-vbbbn,app-sa-4g3nkfre,kind-control-plane/172.18.0.5,Running,registry.k8s.io/ingress-nginx/controller:v1.13.3,Waiting,Terminated,False,1.0,<none>,None,Scheduled,Successfully assigned qa-app/ingress-nginx-4g3...,Name: ingress-nginx-4g3nkfre-656c6...,----------------------------------------------...,----------------------------------------------...,POD DESCRIBE:\nName: ingress-nginx...
3,rbac_forbidden,prod-app,ingress-nginx-n5oblcdc-78d4488dfd-gnt5g,app-sa-n5oblcdc,kind-control-plane/172.18.0.5,Running,registry.k8s.io/ingress-nginx/controller:v1.13.3,Waiting,Terminated,False,1.0,<none>,None,Scheduled,Successfully assigned prod-app/ingress-nginx-n...,Name: ingress-nginx-n5oblcdc-78d44...,----------------------------------------------...,----------------------------------------------...,POD DESCRIBE:\nName: ingress-nginx...
4,rbac_forbidden,deployment,kube-state-metrics-yrnzw8kt-84b9496c49-qtsw4,app-sa-yrnzw8kt,kind-control-plane/172.18.0.5,Running,registry.k8s.io/kube-state-metrics/kube-state-...,Running,None,True,0.0,<none>,None,Scheduled,Successfully assigned deployment/kube-state-me...,Name: kube-state-metrics-yrnzw8kt-...,I0331 06:37:30.542740 1 wrapper.go:120] ...,Error from server (BadRequest): previous termi...,POD DESCRIBE:\nName: kube-state-me...


In [3]:
# Data types and missing values
print("=== Column Info ===")
print(df.dtypes.to_string())
print(f"\n=== Missing Values ===")
print(df.isnull().sum().to_string())
print(f"\nTotal missing: {df.isnull().sum().sum()}")

=== Column Info ===
scenario_id              object
namespace                object
pod_name                 object
service_account_name     object
node                     object
pod_status               object
image                    object
container_state          object
last_state               object
ready                    object
restart_count           float64
node_selectors           object
claim_name               object
event_reason             object
event_message            object
pod_describe             object
pod_logs                 object
pod_logs_previous        object
evidence_text            object

=== Missing Values ===
scenario_id                0
namespace                  0
pod_name                   0
service_account_name       0
node                       0
pod_status                 0
image                      0
container_state         2512
last_state              5748
ready                   2512
restart_count           2512
node_selectors             0


In [4]:
# Rows per scenario_id
print("=== Rows per Scenario ===")
scenario_counts = df['scenario_id'].value_counts()
print(scenario_counts.to_string())
print(f"\nTotal scenarios: {df['scenario_id'].nunique()}")

=== Rows per Scenario ===
scenario_id
pvc_not_found_mountfail                         512
failedscheduling_taint                          500
dns_resolution_failure                          500
service_connection_refused                      500
createcontainerconfigerror_missing_secret       500
createcontainerconfigerror_bad_configmap_key    500
imagepull_bad_tag                               500
crashloop_bad_args                              500
oomkilled_limit_too_low                         500
imagepull_registry_auth                         500
failedscheduling_insufficient_memory            500
failedscheduling_insufficient_cpu               500
nodeselector_mismatch                           500
rbac_forbidden                                  488

Total scenarios: 14


In [5]:
# Unique values per categorical column
categorical_cols = ['scenario_id', 'namespace', 'pod_status', 'container_state', 'last_state', 'ready', 'event_reason']
print("=== Unique Values ===")
for col in categorical_cols:
    print(f"\n{col} ({df[col].nunique()} unique):")
    print(df[col].value_counts().head(10).to_string())

=== Unique Values ===

scenario_id (14 unique):
scenario_id
pvc_not_found_mountfail                         512
failedscheduling_taint                          500
dns_resolution_failure                          500
service_connection_refused                      500
createcontainerconfigerror_missing_secret       500
createcontainerconfigerror_bad_configmap_key    500
imagepull_bad_tag                               500
crashloop_bad_args                              500
oomkilled_limit_too_low                         500
imagepull_registry_auth                         500

namespace (5510 unique):
namespace
monitoring        173
data-pipeline     162
app-deployment    161
app               159
processsing       153
prod-app          152
app-data          152
qa-app            146
deployment        125
metrics           117

pod_status (3 unique):
pod_status
Pending    4512
Running    1488
Failed     1000

container_state (3 unique):
container_state
Waiting       2563
Terminated    169

In [7]:
len(df.columns)

19

In [10]:
df.columns.tolist()

['scenario_id',
 'namespace',
 'pod_name',
 'service_account_name',
 'node',
 'pod_status',
 'image',
 'container_state',
 'last_state',
 'ready',
 'restart_count',
 'node_selectors',
 'claim_name',
 'event_reason',
 'event_message',
 'pod_describe',
 'pod_logs',
 'pod_logs_previous',
 'evidence_text']